# Step 5 — SageMaker Pipeline (Full Orchestration)
Chains preprocessing + KMeans training into one repeatable pipeline.

In [ ]:
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput

role          = sagemaker.get_execution_role()
pipeline_sess = PipelineSession()
BUCKET        = 'hybrid-rec-demo-YOUR_ACCOUNT_ID'   # <-- REPLACE THIS


In [ ]:
# Step A: Preprocessing
processor = SKLearnProcessor(
    framework_version='1.0-1', role=role,
    instance_type='ml.t3.medium', instance_count=1,
    sagemaker_session=pipeline_sess
)

preprocessing_step = ProcessingStep(
    name='FeatureEngineering',
    processor=processor,
    code='../scripts/preprocessing.py',
    inputs=[ProcessingInput(
        source=f's3://{BUCKET}/data/raw/',
        destination='/opt/ml/processing/input'
    )],
    outputs=[ProcessingOutput(
        source='/opt/ml/processing/output',
        destination=f's3://{BUCKET}/data/processed/'
    )]
)


In [ ]:
# Step B: KMeans Training
estimator = SKLearn(
    entry_point='train_kmeans.py', source_dir='../scripts/',
    framework_version='1.0-1', role=role,
    instance_type='ml.t3.medium', instance_count=1,
    hyperparameters={'n_clusters': 4, 'random_state': 42},
    sagemaker_session=pipeline_sess
)

training_step = TrainingStep(
    name='KMeansTraining',
    estimator=estimator,
    inputs={'train': preprocessing_step.properties
                     .ProcessingOutputConfig.Outputs['output-1'].S3Output.S3Uri},
    depends_on=[preprocessing_step]
)


In [ ]:
# Assemble and run pipeline
pipeline = Pipeline(
    name='HybridRecPipeline',
    steps=[preprocessing_step, training_step],
    sagemaker_session=pipeline_sess
)

pipeline.upsert(role_arn=role)
print('Pipeline registered.')

execution = pipeline.start()
print(f'Execution started: {execution.arn}')
execution.wait()
print('Pipeline complete!')
